In [4]:
import cv2
import os
import glob
from ultralytics import YOLO

# 配置参数
VIDEO_DIR = "E:/course/bishe/daima/cv"  # 输入视频目录
SAVE_DIR = "E:/course/bishe/daima/processed_frames"  # 输出帧保存目录
YOLO_MODEL_PATH = "E:/course/bishe/daima/yolov8n-hand.pt"  # YOLO模型路径
FRAME_INTERVAL = 4  # 抽取间隔（每4帧取1帧）
IMG_SIZE = 224  # 输出图像尺寸

# 加载YOLO模型
yolo_model = YOLO(YOLO_MODEL_PATH)

def process_video(video_path, save_dir):
    """处理单个视频并保存手部帧"""
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    save_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        # 每隔FRAME_INTERVAL帧处理一次
        if frame_count % FRAME_INTERVAL == 0:
            # 手部检测
            results = yolo_model(frame, verbose=False)
            
            for result in results:
                for box in result.boxes:
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    hand_crop = frame[y1:y2, x1:x2]
                    
                    if hand_crop.size != 0:
                        # 调整尺寸并保存
                        hand_crop = cv2.resize(hand_crop, (IMG_SIZE, IMG_SIZE))
                        save_path = os.path.join(
                            save_dir,
                            f"{os.path.splitext(os.path.basename(video_path))[0]}_frame{save_count:04d}.jpg"
                        )
                        cv2.imwrite(save_path, hand_crop)
                        save_count += 1
                        print(f"保存帧：{save_path}")
        
        frame_count += 1
    
    cap.release()
    return save_count

def main():
    # 创建保存目录
    os.makedirs(SAVE_DIR, exist_ok=True)
    
    # 获取所有视频文件
    video_paths = glob.glob(os.path.join(VIDEO_DIR, "*.mp4"))
    
    # 处理所有视频
    total_saved = 0
    for video_path in video_paths:
        print(f"处理视频：{video_path}")
        saved = process_video(video_path, SAVE_DIR)
        total_saved += saved
    
    print(f"\n处理完成！共保存 {total_saved} 帧到：{SAVE_DIR}")

if __name__ == "__main__":
    main()

FileNotFoundError: [Errno 2] No such file or directory: 'E:\\course\\bishe\\daima\\yolov8n-hand.pt'